# PDF Loader

In [ ]:
from pathlib import Path
from typing import List

import fitz

from src.models.document import Document, DocumentMetadata

def load_pdf(file_path: Path) -> List[Document]:
    """ 
    Load a PDF file and return a list of Document objects, 
    where each Document represents a single page. 
    """
    pdf_path = Path(file_path)

    # validate file existence
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")
    
    # validate file extension
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a PDF file, got: {pdf_path.suffix}")
    
    documents: List[Document] = []

    # Open pdf
    pdf_document = fitz.open(pdf_path)

    try:
        # iterate through each page
        for page_index in range(len(pdf_document)):
            page = pdf_document[page_index]

            # extract raw text
            text = page.get_text()

            #text_blocks = page.get_text("blocks")
            #text_blocks.sort(key=lambda b: (b[0], b[1]))

            # concatenate text from all blocks
            #text = "\n\n ".join([block[4].strip() for block in text_blocks if block[4].strip()])

            # skip empty pages
            if not text.strip(): 
                continue

            # create metadata object
            metadata = DocumentMetadata(
                source=pdf_path.name,
                file_type="pdf",
                page=page_index + 1,
            )

            # create document object
            document = Document(
                page_content=text,
                metadata=metadata
            )

            documents.append(document)
    
    finally: 
        pdf_document.close()

    return documents 

# PDF Cleaner

In [ ]:
import re
from typing import List
from src.models.document import Document

class TextCleaner:
    def __init__(self):
        # 1. Regex to strip out the web-print date and header boilerplate
        self.header_boilerplate = re.compile(
            r"^\s*\d{1,2}/\d{1,2}/\d{2,4},\s*\d{1,2}:\d{2}\s*(?:AM|PM)\s*|MacBook Air \(13-inch, M5\) - Tech Specs - Apple Support \(IN\)"
        )
        
        # 2. Regex to strip out footer links, URLs, and page counters (e.g., 1/7)
        self.footer_url_pattern = re.compile(r"https://support\.apple\.com/[^\s]+")
        self.page_counter_pattern = re.compile(r"\b\d\s*/\s*\d\b")
        
        # 3. Strip out the parser omission comments
        self.omitted_pictures = re.compile(r"\*\*==>\s*picture\s*\[.*?\]\s*intentionally\s*omitted\s*<==\*\*")
        
        # 4. Dictionary of known corrupted inline broken words found in Apple's specs layout
        self.word_fixes = {
            r"\bAccessi\s+bility\b": "Accessibility",
            r"\bCon\s+fig\s+ure\b": "Configure",
            r"\bcon\s+fig\s+ure\b": "configure",
            r"\bEn\s+viron\s+men\s+tal\b": "Environmental",
            r"\ben\s+viron\s+men\s+tal\b": "environmental",
        }

    def clean_text(self, text: str) -> str:
        """Applies normalization steps to a single string block."""
        if not text:
            return ""

        # Remove image omission tags
        text = self.omitted_pictures.sub("", text)

        # Split lines to clean up per-page headers/footers cleanly
        lines = text.splitlines()
        cleaned_lines = []

        for line in lines:
            # Clean header metadata noise out of the line text
            line_cleaned = self.header_boilerplate.sub("", line)
            line_cleaned = self.footer_url_pattern.sub("", line_cleaned)
            line_cleaned = self.page_counter_pattern.sub("", line_cleaned)
            
            cleaned_lines.append(line_cleaned)

        # Recombine lines
        text = "\n".join(cleaned_lines)

        # Fix specific broken words using regex keys
        for broken_pattern, fixed_word in self.word_fixes.items():
            text = re.sub(broken_pattern, fixed_word, text)

        # Remove web interactive feedback artifact left at the bottom of the scrape
        text = re.sub(r"\*\*Helpful\?\*\*\s*Yes\s*No.*", "", text, flags=re.DOTALL)
        text = re.sub(r"Copyright\s*©\s*2026\s*Apple\s*Inc\..*", "", text, flags=re.DOTALL)

        # Collapse multiple empty newlines down to a max of two to keep markdown paragraphs distinct
        text = re.sub(r"\n{3,}", "\n\n", text)
        
        return text.strip()


def preprocess_documents(documents: List[Document]) -> List[Document]:
    """
    Main entry point for the preprocessing module pipeline.
    Iterates over parsed documents, cleans their content layer, 
    and returns sanitized Document objects ready for chunking.
    """
    cleaner = TextCleaner()
    preprocessed_docs: List[Document] = []

    for doc in documents:
        cleaned_content = cleaner.clean_text(doc.page_content)
        
        # Keep the page object if it still holds functional text data after cleaning
        if cleaned_content:
            # Create a clean instance inheriting metadata parameters
            cleaned_doc = Document(
                page_content=cleaned_content,
                metadata=doc.metadata
            )
            preprocessed_docs.append(cleaned_doc)

    return preprocessed_docs

In [1]:
from langchain_docling.loader import DoclingLoader

loader = DoclingLoader(file_path="../documents/pdfs/MacBook Air (13-inch, M5) - Tech Specs.pdf")

w:\Data-Science\Projects\8.FullStack-RAG-Application-Project-Atman\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
w:\Data-Science\Projects\8.FullStack-RAG-Application-Project-Atman\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dhanush\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either ne

In [4]:
docs = loader.load()

In [16]:
docs[1].page_content

'Configurable to:\nSky\xa0Blue\nSilver\nStarlight\nMidnight\n10-core CPU with 4\xa0super cores and 6\xa0efficiency cores\n8-core GPU, 10-core GPU\nNeural Accelerators\nHardware-accelerated ray tracing\n16-core Neural\xa0Engine\n153GB/s memory bandwidth\nHardware-accelerated H.264, HEVC, ProRes and ProRes\xa0RAW\nVideo decode engine\nVideo encode engine\nProRes encode and decode engine\nAV1 decode\nM5 with 10-core\xa0CPU and 10-core\xa0GPU\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN)'

# Chunker

In [ ]:
from pathlib import Path
from typing import List

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from src.models.document import Document, DocumentMetadata

class MarkdownLayoutSplitter:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        """
        Initializes a structure-aware Markdown splitter tailored for RAG architectures.
        """
        # Define the structural Markdown headers we want to split by
        self.headers_to_split_on = [
            ("#", "title"),
            ("##", "section"),
            ("###", "sub_section")
        ]
        
        self.markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=self.headers_to_split_on,
            strip_headers=False  # Keep headers within text so LLMs maintain semantic context
        )
        
        # Fallback recursive splitter if a single markdown section is larger than our target chunk window
        self.recursive_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def split_documents(self, cleaned_documents: List[Document]) -> List[Document]:
        """
        Processes cleaned source documents, dynamically tracks hierarchical section headings,
        and splits them into text windows ready for vectorization.
        """
        final_chunks: List[Document] = []

        for doc in cleaned_documents:
            # 1. Structural splits based on markdown heading layers
            markdown_sections = self.markdown_splitter.split_text(doc.page_content)

            for section in markdown_sections:
                # Extract headers mapped by MarkdownHeaderTextSplitter
                section_metadata = section.metadata
                current_section_name = section_metadata.get("section", section_metadata.get("title", None))

                # 2. Sub-chunking if a single spec block exceeds the chunk size limit
                sub_text_chunks = self.recursive_splitter.split_text(section.page_content)

                for sub_text in sub_text_chunks:
                    if not sub_text.strip():
                        continue

                    # Create a deep copy of your original document metadata tracking
                    chunk_metadata = DocumentMetadata(
                        source=doc.metadata.source,
                        file_type=doc.metadata.file_type,
                        page=doc.metadata.page,
                        section=current_section_name  # Injects the active header name directly into metadata
                    )

                    final_chunks.append(
                        Document(
                            page_content=sub_text.strip(),
                            metadata=chunk_metadata
                        )
                    )

        return final_chunks